# Phân loại quan điểm trên twitter


## Tóm tắt
Mục tiêu của bài tập này là dự đoán cảm xúc cho bài đăng trên Twitter nhất định bằng Python. Mỗi bài đăng được đánh giá là 1 trong các nhãn: tích cực, tiêu cực và trung tính.

## Các thư viện python
Tiền xử lý dữ liệu sử dụng các thư viện *pandas*, *gensim* and *numpy*, huấn luyện và kiểm tra dùng *scikit-learn*. Vẽ đồ thị sử dụng *plotly*.

In [ ]:
from collections import Counter
import nltk
import pandas as pd
from emoticons import EmoticonDetector
import re as regex
import numpy as np
import plotly
from plotly import graph_objs
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, RandomizedSearchCV
from time import time
import gensim

# cấu hình thư viện vẽ đồ thị
plotly.offline.init_notebook_mode()

## Dữ liệu nguồn
Dữ liệu đầu vào gồm có 2 CSV: `train.csv` (5971 tweets) and `test.csv` (4000 tweets) - một cho huấn luyện và một cho kiểm tra. Định dạng dữ liệu như sau (dữ liệu kiểm tra không có cột Category):


| Id | Category  | Tweet |
|------|------|------|
|   635930169241374720  | neutral | IOS 9 App Transport Security. Mm need to check if my 3rd party network pod supports it |

# Tiền xử lý dữ liệu
## Đọc dữ liệu
        

In [ ]:
class TwitterData_Initialize():
    data = []
    processed_data = []
    wordlist = []

    data_model = None
    data_labels = None
    is_testing = False

    def initialize(self, csv_file, is_testing_set=False, from_cached=None):
        if from_cached is not None:
            self.data_model = pd.read_csv(from_cached)
            return

        self.is_testing = is_testing_set

        if not is_testing_set:
            # đọc dữ liệu từ csv cho tập train, đặt tên các trường dữ liệu là "id", "emotion", "text"
            #### YOUR CODE HERE ####
            self.data = self.data[self.data["emotion"].isin(["positive", "negative", "neutral"])]
        else:
            # đọc dữ liệu từ csv cho tập test, đặt tên các trường dữ liệu là "id","text"
            #### YOUR CODE HERE ####
            not_null_text = 1 ^ pd.isnull(self.data["text"])
            not_null_id = 1 ^ pd.isnull(self.data["id"])
            self.data = self.data.loc[not_null_id & not_null_text, :]

        self.processed_data = self.data
        self.wordlist = []
        self.data_model = None
        self.data_labels = None

Đọc dữ liệu huấn luyện, hiển thị một vài mẫu dữ liệu

In [ ]:
data = TwitterData_Initialize()
# đọc dữ liệu, hiển thị 05 mẫu dữ liệu
#### YOUR CODE HERE ####

,id,emotion,text
0,635769805279248384,negative,Not Available
1,635930169241374720,neutral,IOS 9 App Transport Security. Mm need to check...
2,635950258682523648,neutral,"Mar if you have an iOS device, you should down..."
3,636030803433009153,negative,@jimmie_vanagon my phone does not run on lates...
4,636100906224848896,positive,Not sure how to start your publication on iOS?...


## Phân phối dữ liệu
Thống kê số mẫu trên từng lớp:


In [ ]:
df = data.processed_data
# tính số data samples cho mỗi lớp "negative","neutral","positive"
#### YOUR CODE HERE ####
dist = [
    graph_objs.Bar(
        x=["negative","neutral","positive"],
        y=[neg, neu, pos],
)]
plotly.offline.iplot({"data":dist, "layout":graph_objs.Layout(title="Sentiment type distribution in training set")})

## Các bước tiền xử lý
Mục tiêu của giai đoạn tiền xử lý nhằm tạo biểu diễn **Bag-of-Words** của dữ liệu. Các bước được tiến hành như sau:
1. Làm sạch
  * Xóa URLs
  * Xóa usernames (mentions)
  * Xóa bài đăng với nội dung *Not Available*
  * Xóa ký tự đặc biệt
  * Xóa ký tự số
2. Xử lý văn bản
  * Tách từ
  * Chuyển thành chữ thường
  * Stem
3. Xây dựng danh sách từ cho Bag-of-Words

### Làm sạch
Để làm sạch dữ liệu, chúng ta xây dựng lớp ```TwitterCleanup```. Lớp này bao gồm các phương thức cho phép thực thi tất cả các nội dung đã liệt kê ở trên. Một số công việc chúng ta có thể sử dụng biểu thức chính quy. Lớp này có phương thức ```iterate()``` cho phép thực thi tất cả các phương thức làm sạch theo thứ tự hợp lý.

In [ ]:
class TwitterCleanuper:
    def iterate(self):
        for cleanup_method in [self.remove_urls,
                               self.remove_usernames,
                               self.remove_na,
                               self.remove_special_chars,
                               self.remove_numbers]:
            yield cleanup_method

    @staticmethod
    def remove_by_regex(tweets, regexp):
        tweets.loc[:, "text"].replace(regexp, "", inplace=True)
        return tweets

    def remove_urls(self, tweets):
        #Xóa các urls, nên sử dụng regex
        #### YOUR CODE HERE ####

    def remove_na(self, tweets):
        return tweets[tweets["text"] != "Not Available"]

    def remove_special_chars(self, tweets):  # it unrolls the hashtags to normal words
        for remove in map(lambda r: regex.compile(regex.escape(r)), [",", ":", "\"", "=", "&", ";", "%", "$",
                                                                     "@", "%", "^", "*", "(", ")", "{", "}",
                                                                     "[", "]", "|", "/", "\\", ">", "<", "-",
                                                                     "!", "?", ".", "'",
                                                                     "--", "---", "#"]):
            tweets.loc[:, "text"].replace(remove, "", inplace=True)
        return tweets

    def remove_usernames(self, tweets):
        return TwitterCleanuper.remove_by_regex(tweets, regex.compile(r"@[^\s]+[\s]?"))

    def remove_numbers(self, tweets):
        # xóa các ký tự là số
        #### YOUR CODE HERE ####

Dữ liệu các tweets được làm sạch như sau

In [ ]:
class TwitterData_Cleansing(TwitterData_Initialize):
    def __init__(self, previous):
        self.processed_data = previous.processed_data

    def cleanup(self, cleanuper):
        t = self.processed_data
        for cleanup_method in cleanuper.iterate():
            if not self.is_testing:
                t = cleanup_method(t)
            else:
                if cleanup_method.__name__ != "remove_na":
                    t = cleanup_method(t)

        self.processed_data = t

In [ ]:
data = TwitterData_Cleansing(data)
data.cleanup(TwitterCleanuper())
# hiển thị một vài mẫu dữ liệu để xem kết quả làm sạch
#### YOUR CODE HERE ####

C:\Program Files\Anaconda3\lib\site-packages\pandas\core\generic.py:3443: SettingWithCopyWarning:


A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy



,id,emotion,text
1,635930169241374720,neutral,IOS App Transport Security Mm need to check if...
2,635950258682523648,neutral,Mar if you have an iOS device you should downl...
3,636030803433009153,negative,my phone does not run on latest IOS which may ...
4,636100906224848896,positive,Not sure how to start your publication on iOS ...
5,636176272947744772,neutral,Two Dollar Tuesday is here with Forklift Quick...


### Tách từ & stemming
Để xử lý văn bản, chúng ta sử dụng thư viện ```nltk```. Đầu tiên các tweets được tách từ bằng cách sử dụng ```nlkt.word_tokenize``` và sau đó, stemming được thực hiện bằng **PorterStemmer** bởi các tweets đều viết bằng tiếng Anh.


In [ ]:
class TwitterData_TokenStem(TwitterData_Cleansing):
    def __init__(self, previous):
        self.processed_data = previous.processed_data

    def stem(self, stemmer=nltk.PorterStemmer()):
        def stem_and_join(row):
            row["text"] = list(map(lambda str: stemmer.stem(str.lower()), row["text"]))
            return row

        self.processed_data = self.processed_data.apply(stem_and_join, axis=1)

    def tokenize(self, tokenizer=nltk.word_tokenize):
        def tokenize_row(row):
            row["text"] = tokenizer(row["text"])
            row["tokenized_text"] = [] + row["text"]
            return row

        self.processed_data = self.processed_data.apply(tokenize_row, axis=1)


In [ ]:
data = TwitterData_TokenStem(data)
data.tokenize()
data.stem()
data.processed_data.head(5)

,id,emotion,text,tokenized_text
1,635930169241374720,neutral,"[io, app, transport, secur, mm, need, to, chec...","[IOS, App, Transport, Security, Mm, need, to, ..."
2,635950258682523648,neutral,"[mar, if, you, have, an, io, devic, you, shoul...","[Mar, if, you, have, an, iOS, device, you, sho..."
3,636030803433009153,negative,"[my, phone, doe, not, run, on, latest, io, whi...","[my, phone, does, not, run, on, latest, IOS, w..."
4,636100906224848896,positive,"[not, sure, how, to, start, your, public, on, ...","[Not, sure, how, to, start, your, publication,..."
5,636176272947744772,neutral,"[two, dollar, tuesday, is, here, with, forklif...","[Two, Dollar, Tuesday, is, here, with, Forklif..."


### Xây dựng danh sách từ
Danh sách từ (từ điển) được xây dựng bằng cách đến số lần xuất hiện các từ trong toàn bộ cơ sở dữ liệu.

Trước tiên xây dựng danh sách từ, trước hết chúng ta hiển thị kết quả nếu không lọc:


In [ ]:
words = Counter()
for idx in data.processed_data.index:
    words.update(data.processed_data.loc[idx, "text"])
# hiển thị 5 từ xuất hiện nhiều nhất
words.most_common(5)

[('the', 3744), ('to', 2477), ('i', 1667), ('a', 1620), ('on', 1557)]

Các từ phổ biến nhất thường là các từ stopwords trong tiếng Anh. Tuy nhiên, chúng tôi sẽ lọc chúng ra, vì mục đích của phân tích này là để xác định tình cảm, những từ như "not" và "n't" có thể ảnh hưởng đến nó rất nhiều. Vì lý do này, từ này sẽ được đưa vào danh sách trắng.

In [ ]:
stopwords=nltk.corpus.stopwords.words("english")
whitelist = ["n't", "not"]
for idx, stop_word in enumerate(stopwords):
    if stop_word not in whitelist:
        del words[stop_word]
# hiển thị 5 từ xuất hiện nhiều nhất
#### YOUR CODE HERE ####

[('may', 1027), ('tomorrow', 764), ('day', 526), ('go', 499), ('thi', 495)]

Tuy nhiên, có một số từ xuất hiện quá nhiều lần, chúng có thể bị lọc. Dựa trên một số phân tích, cận dưới được đặt là 3.
Danh sách từ được lưu vào tệp csv, vì vậy các từ tương tự có thể được sử dụng cho tập thử nghiệm.

In [ ]:
class TwitterData_Wordlist(TwitterData_TokenStem):
    def __init__(self, previous):
        self.processed_data = previous.processed_data

    whitelist = ["n't","not"]
    wordlist = []

    def build_wordlist(self, min_occurrences=3, max_occurences=500, stopwords=nltk.corpus.stopwords.words("english"),
                       whitelist=None):
        self.wordlist = []
        whitelist = self.whitelist if whitelist is None else whitelist
        import os
        if os.path.isfile("data\\wordlist.csv"):
            word_df = pd.read_csv("data\\wordlist.csv")
            word_df = word_df[word_df["occurrences"] > min_occurrences]
            self.wordlist = list(word_df.loc[:, "word"])
            return

        words = Counter()
        for idx in self.processed_data.index:
            words.update(self.processed_data.loc[idx, "text"])

        for idx, stop_word in enumerate(stopwords):
            if stop_word not in whitelist:
                del words[stop_word]

        word_df = pd.DataFrame(data={"word": [k for k, v in words.most_common() if min_occurrences < v < max_occurences],
                                     "occurrences": [v for k, v in words.most_common() if min_occurrences < v < max_occurences]},
                               columns=["word", "occurrences"])

        word_df.to_csv("data\\wordlist.csv", index_label="idx")
        self.wordlist = [k for k, v in words.most_common() if min_occurrences < v < max_occurences]


In [ ]:
data = TwitterData_Wordlist(data)
data.build_wordlist()

In [ ]:
words = pd.read_csv("data\\wordlist.csv")
x_words = list(words.loc[0:10,"word"])
x_words.reverse()
y_occ = list(words.loc[0:10,"occurrences"])
y_occ.reverse()

dist = [
    graph_objs.Bar(
        x=y_occ,
        y=x_words,
        orientation="h"
)]
plotly.offline.iplot({"data":dist, "layout":graph_objs.Layout(title="Top words in built wordlist")})

### Bag-of-words
Dữ liệu sẵn sàng chuyển thành biểu diễn bag-of-words.

In [ ]:
class TwitterData_BagOfWords(TwitterData_Wordlist):
    def __init__(self, previous):
        self.processed_data = previous.processed_data
        self.wordlist = previous.wordlist

    def build_data_model(self):
        label_column = []
        if not self.is_testing:
            label_column = ["label"]

        columns = label_column + list(
            map(lambda w: w + "_bow",self.wordlist))
        labels = []
        rows = []
        for idx in self.processed_data.index:
            current_row = []

            if not self.is_testing:
                # add label
                current_label = self.processed_data.loc[idx, "emotion"]
                labels.append(current_label)
                current_row.append(current_label)

            # add bag-of-words
            tokens = set(self.processed_data.loc[idx, "text"])
            for _, word in enumerate(self.wordlist):
                current_row.append(1 if word in tokens else 0)

            rows.append(current_row)

        self.data_model = pd.DataFrame(rows, columns=columns)
        self.data_labels = pd.Series(labels)
        return self.data_model, self.data_labels

- Sinh biểu diễn Bag of words cho các tweets.
- Hiển thị một số mẫu dữ liệu (nhãn, vector biểu diễn cho các tweets)

In [ ]:
# sinh đặc trưng Bag of words cho các tweets
# Hiển thị một vài mẫu dữ liệu để kiểm tra
#### YOUR CODE HERE ####

,label,go_bow,thi_bow,wa_bow,not_bow,im_bow,see_bow,time_bow,get_bow,like_bow,...,topless_bow,flop_bow,scari_bow,attract_bow,pr_bow,sne_bow,harder_bow,sole_bow,rafe_bow,nc_bow
0,neutral,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,neutral,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,negative,0,0,1,1,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
3,positive,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,neutral,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
grouped = bow.groupby(["label"]).sum()
words_to_visualize = []
sentiments = ["positive","negative","neutral"]
#get the most 7 common words for every sentiment
for sentiment in sentiments:
    words = grouped.loc[sentiment,:]
    words.sort_values(inplace=True,ascending=False)
    for w in words.index[:7]:
        if w not in words_to_visualize:
            words_to_visualize.append(w)


#visualize it
plot_data = []
for sentiment in sentiments:
    plot_data.append(graph_objs.Bar(
            x = [w.split("_")[0] for w in words_to_visualize],
            y = [grouped.loc[sentiment,w] for w in words_to_visualize],
            name = sentiment
    ))

plotly.offline.iplot({
        "data":plot_data,
        "layout":graph_objs.Layout(title="Most common words across sentiments")
    })



# Phân loại

In [ ]:
import random
seed = 666
random.seed(seed)

Huấn luyện bộ phân lớp, hiểu thị F1, precision, recall, và accuracy.

In [ ]:
def test_classifier(X_train, y_train, X_test, y_test, classifier):
    log("")
    log("===============================================")
    classifier_name = str(type(classifier).__name__)
    log("Testing " + classifier_name)
    now = time()
    list_of_labels = sorted(list(set(y_train)))
    # huấn luyện mô hình
    model = #### YOUR CODE HERE ####
    log("Learing time {0}s".format(time() - now))
    now = time()
    # dự đoán kết quả trên tập test
    predictions = #### YOUR CODE HERE ####
    log("Predicting time {0}s".format(time() - now))
    # tính toán các độ đo precision, recall, accuracy
    precision = #### YOUR CODE HERE ####
    recall = #### YOUR CODE HERE ####
    accuracy = #### YOUR CODE HERE ####
    f1 = #### YOUR CODE HERE ####
    log("=================== Results ===================")
    log("            Negative     Neutral     Positive")
    log("F1       " + str(f1))
    log("Precision" + str(precision))
    log("Recall   " + str(recall))
    log("Accuracy " + str(accuracy))
    log("===============================================")

    return precision, recall, accuracy, f1

def log(x):
    #can be used to write to log file
    print(x)

# Thực nghiệm 1: BOW + Naive Bayes
Biểu diễn bag-of-words là nhị phân, do đó, Naive Bayes Classifier là một thuật toán tốt để bắt đầu thử nghiệm.

Tập dữ liệu được chia train:test theo tỉ lệ `` 7: 3 ''



In [ ]:
from sklearn.naive_bayes import BernoulliNB
X_train, X_test, y_train, y_test = train_test_split(bow.iloc[:, 1:], bow.iloc[:, 0],
                                                    train_size=0.7, stratify=bow.iloc[:, 0],
                                                    random_state=seed)
precision, recall, accuracy, f1 = test_classifier(X_train, y_train, X_test, y_test, BernoulliNB())


Testing BernoulliNB
Learing time 0.451092004776001s
Predicting time 0.1190040111541748s
=================== Results ===================
            Negative     Neutral     Positive
F1       [ 0.38949672  0.45072993  0.71369782]
Precision[ 0.45408163  0.48431373  0.65906623]
Recall   [ 0.34099617  0.42150171  0.77820513]
Accuracy 0.579594345421


Kết quả với độ chính xác ở mức 58% có vẻ là một kết quả khá tốt đối với thuật toán cơ bản như Naive Bayes (lưu ý rằng bộ phân loại ngẫu nhiên sẽ mang lại kết quả chính xác khoảng 33%). Tuy nhiên, chúng ta mới chỉ tiến hành thực nghiệm 1 lần. Không đảm bảo độ chính xác này trong trường hợp tổng quát.